In [ ]:
import json
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt

N   = 25
IDX = 9

PROJECT_ROOT = Path.cwd().parent
HIST_DIR     = PROJECT_ROOT / "data" / "0" / f"N{N}_2-32_15" / "training_histories"
path = HIST_DIR / f"training_history_{IDX:04d}.json"
print("Cargando:", path)
with open(path) as f:
    hist = json.load(f)

print(f"N={hist['N']}  N_A={hist['N_A']}  T={hist['T']}  chunk_size={hist['chunk_size']}")
print(f"Fases: {hist['phases']}")
print(f"steps: {len(hist['steps'])}   dkl_records: {len(hist['dkl_records'])}")

In [ ]:
# ---- F(global_step) ----
steps  = hist["steps"]
gs     = np.array([s["global_step"] for s in steps])
F      = np.array([s["F"]           for s in steps])
phase  = np.array([s["phase"]       for s in steps])

# Fronteras de fase (índices de global_step donde cambia `phase`)
phase_edges = [gs[i] for i in range(1, len(phase)) if phase[i] != phase[i-1]]

# ---- DKL ----
dkl_recs = hist["dkl_records"]
dkl_gs   = np.array([r["global_step"] for r in dkl_recs])
dkl_val  = np.array([r["dkl"]         for r in dkl_recs])
dkl_tag  = np.array([r["tag"]         for r in dkl_recs])

print("Tags presentes:", sorted(set(dkl_tag.tolist())))
print(f"F_min = {F.min():.4f}   F_max = {F.max():.4f}")

In [ ]:
fig, ax = plt.subplots(figsize=(10, 4))

ax.plot(gs, F, "-", lw=1.2, color="tab:blue", label=r"$F$")

phase_names = [p["method"] for p in hist["phases"]]
bounds = [gs[0]] + phase_edges + [gs[-1] + 1]
for i, (a, b) in enumerate(zip(bounds[:-1], bounds[1:])):
    ax.axvspan(a, b, alpha=0.06,
               color="tab:blue" if phase_names[i]=="swap" else "tab:red")
    ax.text((a + b) / 2, ax.get_ylim()[1], phase_names[i],
            ha="center", va="top", fontsize=10, alpha=0.7)
for b in phase_edges:
    ax.axvline(b, color="k", ls="--", alpha=0.3)

ax.set_xlabel("global step")
ax.set_ylabel(r"$F = E - T\,S_2$")
ax.set_title(f"N={hist['N']}, N_A={hist['N_A']}, T={hist['T']}   "
             f"F_min={F.min():.4f}   idx={IDX}")
ax.grid(True, alpha=0.3)
ax.legend(loc="upper right")
plt.tight_layout()
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(10, 4))

styles = {
    "monitor":           dict(marker="o", s=14, color="tab:orange", label="monitor"),
    "initial_before":    dict(marker="^", s=45, color="tab:purple", label="initial (pre)"),
    "initial_after":     dict(marker="v", s=45, color="tab:cyan",   label="initial (post)"),
    "warm_start_before": dict(marker="s", s=20, color="tab:red",    label="warm_start (pre)"),
    "warm_start_after":  dict(marker="s", s=20, color="tab:green",  label="warm_start (post)"),
}

for tag, st in styles.items():
    m = dkl_tag == tag
    if m.any():
        ax.scatter(dkl_gs[m], dkl_val[m],
                   marker=st["marker"], s=st["s"], color=st["color"],
                   label=st["label"], zorder=3, edgecolors="k", linewidths=0.3)

for b in phase_edges:
    ax.axvline(b, color="k", ls="--", alpha=0.3)

ax.set_yscale("log")
ax.set_xlabel("global step")
ax.set_ylabel(r"$D_{KL}(p_\theta \,\|\, p_R)$")
ax.set_title(f"Fidelidad de la red reversa  (idx={IDX})")
ax.grid(True, alpha=0.3, which="both")
ax.legend(fontsize=8, ncol=2, loc="upper right")
plt.tight_layout()
plt.show()

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

ax = axes[0]
inner = [(r["inner_step"], r["dkl"]) for r in dkl_recs
         if r["tag"] == "initial_train_step"]
if inner:
    inner.sort()
    xs, ys = zip(*inner)
    ax.plot(xs, ys, "-", color="tab:purple", lw=1.2)
    ax.set_title(f"initial_train  (n_steps={len(xs)})")
    ax.set_xlabel("inner step")
    ax.set_ylabel(r"$D_{KL}$")
    ax.set_yscale("log")
    ax.grid(True, alpha=0.3, which="both")

ax = axes[1]
ws_events = {}
for r in dkl_recs:
    if r["tag"] == "warm_start_step":
        ws_events.setdefault(r["global_step"], []).append(
            (r["inner_step"], r["dkl"]))
for gstep, pts in ws_events.items():
    pts.sort()
    xs, ys = zip(*pts)
    ax.plot(xs, ys, "-", lw=1.0, alpha=0.7, label=f"@ step {gstep}")
ax.set_title(f"warm_start  ({len(ws_events)} eventos)")
ax.set_xlabel("inner step")
ax.set_ylabel(r"$D_{KL}$")
ax.set_yscale("log")
ax.grid(True, alpha=0.3, which="both")
if len(ws_events) <= 8:
    ax.legend(fontsize=7, loc="upper right")

plt.tight_layout()
plt.show()

In [ ]:
INDICES = [1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18]

fig, ax = plt.subplots(figsize=(11, 5))
cmap = plt.cm.viridis

for k, idx in enumerate(INDICES):
    p = HIST_DIR / f"training_history_{idx:04d}.json"
    if not p.exists():
        print("falta:", p.name); continue
    with open(p) as f:
        h = json.load(f)
    gs_ = np.array([s["global_step"] for s in h["steps"]])
    F_  = np.array([s["F"]           for s in h["steps"]])
    ax.plot(gs_, F_, lw=1.0, color=cmap(k / max(1, len(INDICES)-1)),
            label=f"idx={idx}  T={h['T']}")

ax.set_xlabel("global step")
ax.set_ylabel(r"$F$")
ax.set_title(f"N={N}  —  curvas de F por temperatura")
ax.grid(True, alpha=0.3)
ax.legend(fontsize=8)
plt.tight_layout()
plt.show()

In [ ]:
import json
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt

# ---- Configuración ----
N    = 25
N_A  = 25

PROJECT_ROOT = Path.cwd().parent
RESULTS_DIR  = PROJECT_ROOT / "data" / "0" / (f"N{N}_2-32_15" if N == N_A else f"N{N}_NA_{N_A}")
results_file = RESULTS_DIR / f"results_N{N}_vs_T.json"

print("Cargando:", results_file)
with open(results_file) as f:
    res = json.load(f)

T  = np.array(res["T"],          dtype=float)
E  = np.array(res["energy"],     dtype=float)
S2 = np.array(res["entropy"],    dtype=float)
F  = np.array(res["free_energy"], dtype=float)

# Orden por T (por si el barrido no se hizo en orden)
order = np.argsort(T)
T, E, S2, F = T[order], E[order], S2[order], F[order]

# Máscara de entradas válidas (puede haber None si algún índice no se rellenó)
valid = ~(np.isnan(E) | np.isnan(S2) | np.isnan(F))
T, E, S2, F = T[valid], E[valid], S2[valid], F[valid]

print(f"{len(T)} temperaturas válidas, rango [{T.min():.2f}, {T.max():.2f}]")

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

# ── Energía ──
ax = axes[0]
ax.plot(T, E, "o-", color="tab:orange", ms=4, lw=1.2)
ax.set_xlabel(r"$T$")
ax.set_ylabel(r"$\langle H \rangle$")
ax.set_title("Energía")
ax.grid(True, alpha=0.3)

# ── Entropía Rényi-2 ──
ax = axes[1]
ax.plot(T, S2, "s-", color="tab:green", ms=4, lw=1.2)
ax.set_xlabel(r"$T$")
ax.set_ylabel(r"$S_2$")
ax.set_title(r"Entropía Rényi-2")
ax.grid(True, alpha=0.3)

# ── Energía libre ──
ax = axes[2]
ax.plot(T, F, "d-", color="tab:blue", ms=4, lw=1.2)
ax.set_xlabel(r"$T$")
ax.set_ylabel(r"$F = E - T\,S_2$")
ax.set_title("Energía libre")
ax.grid(True, alpha=0.3)

fig.suptitle(f"N = {N}, N_A = {N_A}", y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
import json
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt
from scipy.optimize import brentq

# ── Necesita: exact_dos_tfim(N, J, h, dE) y exact_tfim_thermo(N, J, h, beta)
#    tal como las definiste en celdas anteriores. ────────────────────────

def exact_dos_tfim(N, J, h, dE=0.005):
    """
    DOS exacta (free-fermion, JW) para TFIM 1D. Válida para h > J.
    Devuelve (E_per_site, g_densidad, dE_site) con la misma convención
    que jw_dos_saddle: E en unidades per-site y sum(g)*dE == 1.
    """
    eps_fn = lambda k: 2*np.sqrt(J**2 + h**2 - 2*J*h*np.cos(k))
    sectors = [
        (2*np.pi*(np.arange(N)+0.5)/N, 0),   # NS, paridad par
        (2*np.pi*np.arange(N)/N,       1),   # R,  paridad impar (h > J)
    ]

    E_all, g_all = [], []
    for k, par in sectors:
        e = eps_fn(k)
        nb = int(e.sum()/dE) + 3
        g = np.zeros((2, nb)); g[0, 0] = 1.0
        for ej in e:
            s = ej/dE; lo = int(s); f = s - lo
            new = g.copy()
            for p in (0, 1):
                new[1-p, lo:]   += (1-f)*g[p, :nb-lo]
                new[1-p, lo+1:] +=  f   *g[p, :nb-lo-1]
            g = new
        E_all.append(-0.5*e.sum() + dE*np.arange(nb))
        g_all.append(g[par])

    # Unir los dos sectores en una rejilla común
    E_flat = np.concatenate(E_all)
    g_flat = np.concatenate(g_all)
    E_grid = np.arange(E_flat.min(), E_flat.max() + 1.5*dE, dE)
    g_grid, _ = np.histogram(
        E_flat,
        bins=np.append(E_grid - dE/2, E_grid[-1] + dE/2),
        weights=g_flat,
    )

    # Conversión a per-site + normalización a densidad
    E_grid_site = E_grid / N
    dE_site     = dE / N
    g_grid      = g_grid / (g_grid.sum() * dE_site)

    return E_grid_site, g_grid, dE_site

  
# ---- Configuración ----
N    = 8
N_A  = 8
J_ZZ = -1.0
h_x  = -1.5
J, h = abs(J_ZZ), abs(h_x)

PROJECT_ROOT = Path.cwd().parent
RESULTS_DIR  = PROJECT_ROOT / "data" / "0" / (f"N{N}" if N == N_A else f"N{N}_NA_{N_A}")
results_file = RESULTS_DIR / f"results_N{N}_vs_T.json"

print("Cargando:", results_file)
with open(results_file) as f:
    res = json.load(f)

T  = np.array(res["T"],          dtype=float)
E  = np.array(res["energy"],     dtype=float)   # totales
S2 = np.array(res["entropy"],    dtype=float)   # adimensional
F  = np.array(res["free_energy"], dtype=float)  # totales

order = np.argsort(T)
T, E, S2, F = T[order], E[order], S2[order], F[order]
valid = ~(np.isnan(E) | np.isnan(S2) | np.isnan(F))
T, E, S2, F = T[valid], E[valid], S2[valid], F[valid]
print(f"{len(T)} temperaturas válidas, rango [{T.min():.2f}, {T.max():.2f}]")

# ═════════════════════════════════════════════════════════════════════════
# 1. DOS exacta (free-fermion) — per-site
# ═════════════════════════════════════════════════════════════════════════
dE_site  = 0.005
E_dos, g_dos, dE_dos = exact_dos_tfim(N, J, h, dE=dE_site * N)

# ═════════════════════════════════════════════════════════════════════════
# 2. Termodinámica exacta del MRE (α=2) a partir de la DOS
#    ρ₂ ∝ Π_{E_⊥}(1 - β/2 (H-Ē)) Π_{E_⊥},  E_⊥ = Ē + 2T  (total)
#    per-site: c = ē + 2T/N
# ═════════════════════════════════════════════════════════════════════════
def mre_thermo_exact(T, N, E_dos, g_dos, dE_dos):
    """Devuelve (E_total, S2, F_total) del MRE exacto a temperatura T."""
    if T < 1e-6:
        E_gs = N * E_dos.min()
        return E_gs, 0.0, E_gs

    def mean_E_site(c):
        w = np.maximum(c - E_dos, 0.0)
        Z = np.sum(g_dos * w) * dE_dos
        if Z < 1e-300:
            return np.nan
        return np.sum(g_dos * w * E_dos) * dE_dos / Z

    def self_cons(c):
        return c - mean_E_site(c) - 2 * T / N

    try:
        c = brentq(self_cons,
                   E_dos.min() + 1e-6,
                   E_dos.max() + 20 * T / N + 10,
                   xtol=1e-10)
    except ValueError:
        return np.nan, np.nan, np.nan

    e_bar = mean_E_site(c)
    w     = np.maximum(c - E_dos, 0.0)
    Z     = np.sum(g_dos * w) * dE_dos
    numer = np.sum(g_dos * w**2) * dE_dos
    if numer <= 0 or Z <= 0:
        return np.nan, np.nan, np.nan

    # Tr(ρ²) = numer / (2^N · Z²)   [g_dos está normalizada a ∫g dE = 1]
    log_purity = np.log(numer) - N * np.log(2) - 2 * np.log(Z)
    S2 = -log_purity

    E_tot = N * e_bar
    return E_tot, S2, E_tot - T * S2

def exact_tfim_thermo(N, J, h, beta):
    """Energía y entropía exactas incluyendo los sectores NS y R."""
    k_ns = 2 * np.pi * (np.arange(N) + 0.5) / N
    eps_ns = 2 * np.sqrt(J**2 + h**2 - 2 * J * h * np.cos(k_ns))
    
    k_r = 2 * np.pi * np.arange(N) / N
    eps_r = 2 * np.sqrt(J**2 + h**2 - 2 * J * h * np.cos(k_r))
    
    log_Z_ns = np.sum(np.log(2 * np.cosh(beta * eps_ns / 2)))
    log_Z_r  = np.sum(np.log(2 * np.cosh(beta * eps_r / 2)))
    
    log_Z_sum = np.logaddexp(log_Z_ns, log_Z_r)
    log_Z_tot = log_Z_sum - np.log(2)
    
    P_ns = np.exp(log_Z_ns - log_Z_sum)
    P_r  = np.exp(log_Z_r - log_Z_sum)
    
    E_ns = -np.sum((eps_ns / 2) * np.tanh(beta * eps_ns / 2))
    E_r  = -np.sum((eps_r / 2) * np.tanh(beta * eps_r / 2))
    
    E_tot = P_ns * E_ns + P_r * E_r
    S_tot = log_Z_tot + beta * E_tot
    
    return E_tot / N, S_tot / N

# ═════════════════════════════════════════════════════════════════════════
# 3. Gibbs exacto (fórmula Jordan-Wigner)
# ═════════════════════════════════════════════════════════════════════════
def gibbs_thermo_exact(T, N, J, h):
    """Devuelve (E_total, S_vN, F_total) del Gibbs exacto."""
    if T < 0.02:
        eps_fn = lambda k: 2 * np.sqrt(J**2 + h**2 - 2 * J * h * np.cos(k))
        eps_ns = eps_fn(2 * np.pi * (np.arange(N) + 0.5) / N)
        E_gs   = -0.5 * eps_ns.sum()
        return E_gs, 0.0, E_gs
    E_site, S_site = exact_tfim_thermo(N, J, h, 1.0 / T)
    E_tot = N * E_site
    S_tot = N * S_site
    return E_tot, S_tot, E_tot - T * S_tot

# ═════════════════════════════════════════════════════════════════════════
# 4. Curvas de referencia sobre una rejilla fina
# ═════════════════════════════════════════════════════════════════════════
T_ref = np.linspace(max(T.min(), 0.05), T.max() * 1.05, 200)

E_mre   = np.zeros_like(T_ref); S2_mre  = np.zeros_like(T_ref); F_mre  = np.zeros_like(T_ref)
E_gib   = np.zeros_like(T_ref); SvN_gib = np.zeros_like(T_ref); F_gib  = np.zeros_like(T_ref)

for i, Ti in enumerate(T_ref):
    E_mre[i], S2_mre[i], F_mre[i] = mre_thermo_exact(Ti, N, E_dos, g_dos, dE_dos)
    E_gib[i], SvN_gib[i], F_gib[i] = gibbs_thermo_exact(Ti, N, J, h)

# Interpolamos a las T del vstate para el gap
F_mre_at_T  = np.interp(T, T_ref, F_mre)
E_mre_at_T  = np.interp(T, T_ref, E_mre)
S2_mre_at_T = np.interp(T, T_ref, S2_mre)

# ═════════════════════════════════════════════════════════════════════════
# 5. Plot 2×2
# ═════════════════════════════════════════════════════════════════════════
fig, axes = plt.subplots(2, 2, figsize=(13, 9))

# ── Energía (per-site) ──
ax = axes[0, 0]
ax.plot(T, E, "o", color="tab:orange", ms=4, label="vstate (trained)")
ax.plot(T_ref, E_mre, "-",  color="tab:red",  lw=2, label="MRE exacto")
ax.plot(T_ref, E_gib, "--", color="tab:blue", lw=2, label="Gibbs exacto")
ax.set_xlabel(r"$T$"); ax.set_ylabel(r"$\langle H \rangle$")
ax.set_title("Energía"); ax.legend(fontsize=9); ax.grid(alpha=0.3)

# ── Entropía ──
ax = axes[0, 1]
ax.plot(T, S2, "s", color="tab:green", ms=4, label=r"$S_2$ vstate (trained)")
ax.plot(T_ref, S2_mre, "-", color="tab:red", lw=2, label=r"$S_2$ MRE exacto")
ax.plot(T_ref, SvN_gib, "--", color="tab:blue", lw=2, label=r"$S_{\rm vN}$ Gibbs")
ax.axhline(N * np.log(2), color="gray", ls=":", lw=1, label=r"$N\ln 2$")
ax.set_xlabel(r"$T$"); ax.set_ylabel("Entropía")
ax.set_title("Entropía"); ax.legend(fontsize=9); ax.grid(alpha=0.3)

# ── Energía libre (per-site) ──
ax = axes[1, 0]
ax.plot(T, F, "d", color="tab:blue", ms=4, label="vstate (trained)")
ax.plot(T_ref, F_mre, "-",  color="tab:red",  lw=2, label=r"$F$ MRE exacto")
ax.plot(T_ref, F_gib , "--", color="tab:gray", lw=2, label=r"$F$ Gibbs exacto")
ax.set_xlabel(r"$T$"); ax.set_ylabel(r"$F/N = (E - T\,S)/N$")
ax.set_title("Energía libre"); ax.legend(fontsize=9); ax.grid(alpha=0.3)

# ── Gap diagnóstico ──
ax = axes[1, 1]
gap = F - F_mre_at_T
ax.axhline(0, color="k", ls="--", lw=1)
ax.plot(T, gap, "o-", color="tab:purple", ms=5, lw=1.2)
ax.set_xlabel(r"$T$"); ax.set_ylabel(r"$F_{\rm vstate} - F_{\rm MRE}$  (absoluto)")
ax.set_title("Gap vs MRE exacto")
ax.grid(alpha=0.3)

ax2 = ax.twinx()
gap_rel = gap / np.maximum(np.abs(F_mre_at_T), 1e-12)
ax2.plot(T, gap_rel, "s", color="tab:brown", ms=3, alpha=0.6)
ax2.set_ylabel(r"gap relativo", color="tab:brown")
ax2.tick_params(axis='y', labelcolor="tab:brown")
ax2.axhline(0, color="brown", ls=":", lw=0.5, alpha=0.5)

fig.suptitle(f"N = {N}, N_A = {N_A}", y=1.00, fontsize=13)
plt.tight_layout()
plt.show()